# ACIS Insurance Risk Analytics: Predictive Modeling

This notebook builds claim-severity models for AlphaCare Insurance Solutions (ACIS).

The goal is to compare Linear Regression, Random Forest, and XGBoost models and determine whether claim severity can be predicted well enough to support risk-based pricing decisions.

## 1. Setup

Import project utilities and modeling dependencies. The reusable functions in `src.modeling` handle preprocessing, missing values, categorical encoding, model training, and evaluation.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_insurance_data, summarize_dataset
from src.model_interpretability import (
    compute_shap_values,
    save_shap_summary_plot,
    shap_feature_importance,
)
from src.modeling import (
    build_linear_model,
    build_random_forest_model,
    build_xgboost_model,
    metrics_to_frame,
    train_model,
)

RANDOM_STATE = 42
TEST_SIZE = 0.2

sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

## 2. Load Cleaned Data

Use the DVC-generated cleaned dataset from `data/processed/insurance_data_cleaned.csv`.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "insurance_data_cleaned.csv"

df = load_insurance_data(DATA_PATH)
summarize_dataset(df)

**Business interpretation:** Modeling uses the cleaned, reproducible dataset so model results can be audited and regenerated from the DVC pipeline.

## 3. Define the Severity Modeling Dataset

Claim severity usually means the claim amount for policies that actually had a claim. For this project, `TotalClaims` is used as the severity target, and the modeling dataset is filtered to rows where `TotalClaims > 0`.

This avoids training a severity model that is dominated by zero-claim policies. Claim occurrence can be handled separately as a classification problem.

In [ ]:
TARGET_COL = "TotalClaims"

candidate_features = [
    "Age",
    "Gender",
    "Province",
    "VehicleType",
    "AnnualIncome",
    "RiskScore",
    "AnnualPremium",
    "Deductible",
    "NCD",
    "PastClaims",
    "TotalPremium",
    "CoverType",
    "AutoMake",
    "VehicleModel",
    "CustomValueEstimate",
    "ZipCode",
]

feature_cols = [column for column in candidate_features if column in df.columns]
severity_df = df[df[TARGET_COL] > 0].copy()

print(f"Severity modeling rows: {len(severity_df):,}")
print(f"Selected features: {feature_cols}")
severity_df[[TARGET_COL] + feature_cols].head()

In [ ]:
severity_df[TARGET_COL].describe()

**Business interpretation:** The model is trained only on claim records, so the prediction target represents expected claim severity once a claim has occurred. This can support pricing decisions when combined with a separate claim-frequency model.

## 4. Train Linear Regression

Linear Regression provides a simple baseline. It is easy to explain, but it may underperform if claim severity has nonlinear relationships with vehicle, customer, or policy features.

In [ ]:
linear_result = train_model(
    severity_df,
    target_col=TARGET_COL,
    model=build_linear_model("regression"),
    feature_cols=feature_cols,
    task_type="regression",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    scale_numeric=True,
)

linear_metrics = metrics_to_frame(
    "Linear Regression",
    linear_result["metrics"],
    task_type="regression",
)
linear_metrics

## 5. Train Random Forest

Random Forest can capture nonlinear relationships and interactions without requiring heavy feature engineering. It is often a strong benchmark for tabular insurance data.

In [ ]:
random_forest_result = train_model(
    severity_df,
    target_col=TARGET_COL,
    model=build_random_forest_model(
        "regression",
        n_estimators=200,
        max_depth=8,
        min_samples_leaf=5,
        n_jobs=-1,
    ),
    feature_cols=feature_cols,
    task_type="regression",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

random_forest_metrics = metrics_to_frame(
    "Random Forest",
    random_forest_result["metrics"],
    task_type="regression",
)
random_forest_metrics

## 6. Train XGBoost

XGBoost is a gradient-boosted tree model that often performs well on structured tabular data. It can capture complex relationships while controlling overfitting through tree depth, learning rate, and regularization.

In [ ]:
xgboost_result = train_model(
    severity_df,
    target_col=TARGET_COL,
    model=build_xgboost_model(
        "regression",
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        n_jobs=-1,
    ),
    feature_cols=feature_cols,
    task_type="regression",
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

xgboost_metrics = metrics_to_frame(
    "XGBoost",
    xgboost_result["metrics"],
    task_type="regression",
)
xgboost_metrics

## 7. Model Comparison

Compare models using RMSE and R².

- Lower RMSE means smaller average prediction error in claim currency units.
- Higher R² means the model explains more variation in claim severity.

In [ ]:
model_comparison = pd.concat(
    [linear_metrics, random_forest_metrics, xgboost_metrics],
    ignore_index=True,
)

model_comparison = model_comparison.sort_values(
    by=["rmse", "r2"],
    ascending=[True, False],
).reset_index(drop=True)

model_comparison

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=model_comparison, x="model_name", y="rmse")
plt.title("Model Comparison by RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=20, ha="right")
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=model_comparison, x="model_name", y="r2")
plt.title("Model Comparison by R²")
plt.xlabel("Model")
plt.ylabel("R²")
plt.xticks(rotation=20, ha="right")
plt.show()

In [ ]:
best_model_row = model_comparison.iloc[0]
best_model_name = best_model_row["model_name"]

print(
    f"Best model by RMSE: {best_model_name} "
    f"(RMSE={best_model_row['rmse']:.4f}, R²={best_model_row['r2']:.4f})"
)

## 8. Which Model Performs Best?

The best model is selected primarily by the lowest RMSE because ACIS needs severity predictions with the smallest claim-amount error. R² is used as a secondary measure to understand how much variation in claim severity the model explains.

If Random Forest or XGBoost performs best, it likely means claim severity depends on nonlinear relationships or interactions between customer, vehicle, policy, and location features. If Linear Regression performs similarly, the simpler model may be preferred because it is easier to explain and audit.

## 9. Business Interpretation

A useful severity model can help ACIS estimate expected claim cost after a claim occurs. This supports risk-based pricing by helping the company understand which policies may require higher premium adequacy.

Important business considerations:

- RMSE should be interpreted in claim-currency units, so lower values mean smaller pricing error.
- A low or negative R² means the current feature set does not explain severity well enough for confident automated pricing.
- Tree-based models may improve predictive power, but ACIS should also review feature importance and regulatory constraints before operational use.
- Severity modeling should be combined with claim-frequency modeling to estimate full expected loss.

## 10. Model Interpretability with SHAP

SHAP explains how each feature contributes to the best model's claim severity predictions. The table below ranks the top features by average absolute SHAP value, which means the features with the largest overall impact on predicted severity appear first.

In [ ]:
model_results = {
    "Linear Regression": linear_result,
    "Random Forest": random_forest_result,
    "XGBoost": xgboost_result,
}

best_result = model_results[best_model_name]

shap_values, shap_X = compute_shap_values(
    best_result["model"],
    best_result["X_test"],
    max_rows=500,
    random_state=RANDOM_STATE,
)

top_features = shap_feature_importance(
    shap_values,
    shap_X,
    top_n=10,
)

top_features

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

shap_plot_path = REPORTS_DIR / "shap_summary_claim_severity.png"
save_shap_summary_plot(
    shap_values,
    shap_X,
    shap_plot_path,
    max_display=10,
)

print(f"Saved SHAP summary plot to {shap_plot_path}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top_features.sort_values("mean_abs_shap", ascending=True),
    x="mean_abs_shap",
    y="feature_label",
)
plt.title(f"Top SHAP Features for {best_model_name}")
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

**Business interpretation:** The top SHAP features show which customer, vehicle, policy, and location attributes most influence predicted claim severity. A positive SHAP value increases the predicted claim amount, while a negative SHAP value decreases it. ACIS should use these features to understand pricing drivers, but final pricing decisions should also consider regulatory constraints, fairness, and actuarial review.

## 11. Report-Ready Feature Explanations

The following table is suitable for the final report. It gives the top 5 to 10 influential features and a business-facing explanation of why each feature matters.

In [ ]:
report_feature_explanations = top_features[
    ["feature_label", "mean_abs_shap", "business_explanation"]
].rename(
    columns={
        "feature_label": "feature",
        "mean_abs_shap": "importance",
        "business_explanation": "business_interpretation",
    }
)

report_feature_explanations.to_csv(
    REPORTS_DIR / "model_interpretability_top_features.csv",
    index=False,
)

report_feature_explanations

## 12. Save Model Comparison Results

Save the model comparison table for reuse in the final report.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)

model_comparison.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
model_comparison

## 13. Next Steps

Recommended next steps for ACIS:

- Add claim-frequency classification to estimate the probability of a claim.
- Combine frequency and severity predictions into expected loss.
- Use SHAP outputs to explain the best model in the final report.
- Validate model stability with cross-validation.
- Review pricing recommendations with business, actuarial, and compliance stakeholders.